In [3]:
from smartcard.Exceptions import NoCardException
from smartcard.System import readers
from smartcard.util import toHexString

for reader in readers():
    try:
        connection = reader.createConnection()
        connection.connect()
        print(reader, toHexString(connection.getATR()))
    except NoCardException:
        print(reader, "no card inserted")

Alcor Micro AU9540 00 00 no card inserted
REINER SCT cyberJack RFID basis 01 00 3B 81 80 01 80 80


In [6]:
import struct
import sys

import smartcard.util
from smartcard.scard import *

attributes = {
    SCARD_ATTR_ATR_STRING: "SCARD_ATTR_ATR_STRING",
    SCARD_ATTR_CHANNEL_ID: "SCARD_ATTR_CHANNEL_ID",
    SCARD_ATTR_CHARACTERISTICS: "SCARD_ATTR_CHARACTERISTICS",
    SCARD_ATTR_CURRENT_BWT: "SCARD_ATTR_CURRENT_BWT",
    SCARD_ATTR_CURRENT_CLK: "SCARD_ATTR_CURRENT_CLK",
    SCARD_ATTR_CURRENT_CWT: "SCARD_ATTR_CURRENT_CWT",
    SCARD_ATTR_CURRENT_D: "SCARD_ATTR_CURRENT_D",
    SCARD_ATTR_CURRENT_EBC_ENCODING: "SCARD_ATTR_CURRENT_EBC_ENCODING",
    SCARD_ATTR_CURRENT_F: "SCARD_ATTR_CURRENT_F",
    SCARD_ATTR_CURRENT_IFSC: "SCARD_ATTR_CURRENT_IFSC",
    SCARD_ATTR_CURRENT_IFSD: "SCARD_ATTR_CURRENT_IFSD",
    SCARD_ATTR_CURRENT_IO_STATE: "SCARD_ATTR_CURRENT_IO_STATE",
    SCARD_ATTR_CURRENT_N: "SCARD_ATTR_CURRENT_N",
    SCARD_ATTR_CURRENT_PROTOCOL_TYPE: "SCARD_ATTR_CURRENT_PROTOCOL_TYPE",
    SCARD_ATTR_CURRENT_W: "SCARD_ATTR_CURRENT_W",
    SCARD_ATTR_DEFAULT_CLK: "SCARD_ATTR_DEFAULT_CLK",
    SCARD_ATTR_DEFAULT_DATA_RATE: "SCARD_ATTR_DEFAULT_DATA_RATE",
    SCARD_ATTR_DEVICE_FRIENDLY_NAME_A: "SCARD_ATTR_DEVICE_FRIENDLY_NAME_A",
    SCARD_ATTR_DEVICE_FRIENDLY_NAME_W: "SCARD_ATTR_DEVICE_FRIENDLY_NAME_W",
    SCARD_ATTR_DEVICE_IN_USE: "SCARD_ATTR_DEVICE_IN_USE",
    SCARD_ATTR_DEVICE_SYSTEM_NAME_A: "SCARD_ATTR_DEVICE_SYSTEM_NAME_A",
    SCARD_ATTR_DEVICE_SYSTEM_NAME_W: "SCARD_ATTR_DEVICE_SYSTEM_NAME_W",
    SCARD_ATTR_DEVICE_UNIT: "SCARD_ATTR_DEVICE_UNIT",
    SCARD_ATTR_ESC_AUTHREQUEST: "SCARD_ATTR_ESC_AUTHREQUEST",
    SCARD_ATTR_ESC_CANCEL: "SCARD_ATTR_ESC_CANCEL",
    SCARD_ATTR_ESC_RESET: "SCARD_ATTR_ESC_RESET",
    SCARD_ATTR_EXTENDED_BWT: "SCARD_ATTR_EXTENDED_BWT",
    SCARD_ATTR_ICC_INTERFACE_STATUS: "SCARD_ATTR_ICC_INTERFACE_STATUS",
    SCARD_ATTR_ICC_PRESENCE: "SCARD_ATTR_ICC_PRESENCE",
    SCARD_ATTR_ICC_TYPE_PER_ATR: "SCARD_ATTR_ICC_TYPE_PER_ATR",
    SCARD_ATTR_MAXINPUT: "SCARD_ATTR_MAXINPUT",
    SCARD_ATTR_MAX_CLK: "SCARD_ATTR_MAX_CLK",
    SCARD_ATTR_MAX_DATA_RATE: "SCARD_ATTR_MAX_DATA_RATE",
    SCARD_ATTR_MAX_IFSD: "SCARD_ATTR_MAX_IFSD",
    SCARD_ATTR_POWER_MGMT_SUPPORT: "SCARD_ATTR_POWER_MGMT_SUPPORT",
    SCARD_ATTR_SUPRESS_T1_IFS_REQUEST: "SCARD_ATTR_SUPRESS_T1_IFS_REQUEST",
    SCARD_ATTR_USER_AUTH_INPUT_DEVICE: "SCARD_ATTR_USER_AUTH_INPUT_DEVICE",
    SCARD_ATTR_USER_TO_CARD_AUTH_DEVICE: "SCARD_ATTR_USER_TO_CARD_AUTH_DEVICE",
    SCARD_ATTR_VENDOR_IFD_SERIAL_NO: "SCARD_ATTR_VENDOR_IFD_SERIAL_NO",
    SCARD_ATTR_VENDOR_IFD_TYPE: "SCARD_ATTR_VENDOR_IFD_TYPE",
    SCARD_ATTR_VENDOR_IFD_VERSION: "SCARD_ATTR_VENDOR_IFD_VERSION",
    SCARD_ATTR_VENDOR_NAME: "SCARD_ATTR_VENDOR_NAME",
}
if "pcsclite" == resourceManager:
    extra_attributes = {
        SCARD_ATTR_ASYNC_PROTOCOL_TYPES: "SCARD_ATTR_ASYNC_PROTOCOL_TYPES",
        SCARD_ATTR_SYNC_PROTOCOL_TYPES: "SCARD_ATTR_SYNC_PROTOCOL_TYPES",
    }
    attributes.update(extra_attributes)


def printAttribute(attrib, value):
    print("-----------------", attributes[attrib], "-----------------")
    print(value)
    print(smartcard.util.toHexString(value, smartcard.util.HEX))
    print(struct.pack(*["<" + "B" * len(value)] + value))


if __name__ == "__main__":
    hresult, hcontext = SCardEstablishContext(SCARD_SCOPE_USER)
    if hresult != SCARD_S_SUCCESS:
        raise error("Failed to establish context: " + SCardGetErrorMessage(hresult))
    print("Context established!")

    try:
        hresult, readers = SCardListReaders(hcontext, [])
        if hresult != SCARD_S_SUCCESS:
            raise error("Failed to list readers: " + SCardGetErrorMessage(hresult))
        print("PCSC Readers:", readers)

        if len(readers) < 1:
            raise error("No smart card readers")

        for reader in readers:
            print("Trying to retrieve attributes of", reader)
            hresult, hcard, dwActiveProtocol = SCardConnect(
                hcontext,
                reader,
                SCARD_SHARE_SHARED,
                SCARD_PROTOCOL_T0 | SCARD_PROTOCOL_T1,
            )
            if hresult != SCARD_S_SUCCESS:
                print(error, "Unable to connect: " + SCardGetErrorMessage(hresult))
            else:

                print("Connected with active protocol", dwActiveProtocol)

                try:
                    for i in list(attributes.keys()):
                        hresult, attrib = SCardGetAttrib(hcard, i)
                        if hresult == SCARD_S_SUCCESS:
                            printAttribute(i, attrib)
                        else:
                            print(
                                "-----------------", attributes[i], "-----------------"
                            )
                            print("error:", SCardGetErrorMessage(hresult))

                finally:
                    hresult = SCardDisconnect(hcard, SCARD_UNPOWER_CARD)
                    if hresult != SCARD_S_SUCCESS:
                        raise error(
                            "Failed to disconnect: " + SCardGetErrorMessage(hresult)
                        )
                    print("Disconnected")

    finally:
        hresult = SCardReleaseContext(hcontext)
        if hresult != SCARD_S_SUCCESS:
            raise error("Failed to release context: " + SCardGetErrorMessage(hresult))
        print("Released context.")

Context established!
PCSC Readers: ['Alcor Micro AU9540 00 00', 'REINER SCT cyberJack RFID basis 01 00']
Trying to retrieve attributes of Alcor Micro AU9540 00 00
<class 'scard.error'> Unable to connect: No smart card inserted.
Trying to retrieve attributes of REINER SCT cyberJack RFID basis 01 00
Connected with active protocol 2
----------------- SCARD_ATTR_ATR_STRING -----------------
[59, 129, 128, 1, 128, 128]
0x3B 0x81 0x80 0x01 0x80 0x80
b';\x81\x80\x01\x80\x80'
----------------- SCARD_ATTR_CHANNEL_ID -----------------
[16, 1, 32, 0]
0x10 0x01 0x20 0x00
b'\x10\x01 \x00'
----------------- SCARD_ATTR_CHARACTERISTICS -----------------
error: Feature not supported.
----------------- SCARD_ATTR_CURRENT_BWT -----------------
error: Feature not supported.
----------------- SCARD_ATTR_CURRENT_CLK -----------------
error: Feature not supported.
----------------- SCARD_ATTR_CURRENT_CWT -----------------
error: Feature not supported.
----------------- SCARD_ATTR_CURRENT_D -----------------


In [7]:
import platform
import sys

import smartcard.guid
from smartcard.scard import *

if "winscard" == resourceManager:

    znewcardName = "dummy-card"
    znewcardATR = [
        0x3B,
        0x77,
        0x94,
        0x00,
        0x00,
        0x82,
        0x30,
        0x00,
        0x13,
        0x6C,
        0x9F,
        0x22,
    ]
    znewcardMask = [
        0xFF,
        0xFF,
        0xFF,
        0xFF,
        0xFF,
        0xFF,
        0xFF,
        0xFF,
        0xFF,
        0xFF,
        0xFF,
        0xFF,
    ]
    znewcardPrimGuid = smartcard.guid.strToGUID(
        "{128F3806-4F70-4ccf-977A-60C390664840}"
    )
    znewcardSecGuid = smartcard.guid.strToGUID("{EB7F69EA-BA20-47d0-8C50-11CFDEB63BBE}")

    def main():
        hresult, hcontext = SCardEstablishContext(SCARD_SCOPE_USER)
        if hresult != SCARD_S_SUCCESS:
            raise scard.error(
                "Failed to establish context: " + SCardGetErrorMessage(hresult)
            )
        print("Context established!")

        try:

            # list interfaces for a known card
            expectedCard = "Schlumberger Cryptoflex 8k v2"
            hresult, interfaces = SCardListInterfaces(hcontext, expectedCard)
            if hresult != SCARD_S_SUCCESS:
                raise scard.error(
                    "Failed to list interfaces: " + SCardGetErrorMessage(hresult)
                )
            print("Interfaces for ", expectedCard, ":", interfaces)

            # introduce a card (forget first in case it is already present)
            hresult = SCardForgetCardType(hcontext, znewcardName)
            print("Introducing card " + znewcardName)
            hresult = SCardIntroduceCardType(
                hcontext,
                znewcardName,
                znewcardPrimGuid,
                znewcardPrimGuid + znewcardSecGuid,
                znewcardATR,
                znewcardMask,
            )
            if hresult != SCARD_S_SUCCESS:
                raise error(
                    "Failed to introduce card type: " + SCardGetErrorMessage(hresult)
                )

            # list card interfaces
            hresult, interfaces = SCardListInterfaces(hcontext, znewcardName)
            if hresult != SCARD_S_SUCCESS:
                raise error(
                    "Failed to list interfaces: " + SCardGetErrorMessage(hresult)
                )
            for i in interfaces:
                print(
                    "Interface for " + znewcardName + " :", smartcard.guid.GUIDToStr(i)
                )

            print("Forgetting card " + znewcardName)
            hresult = SCardForgetCardType(hcontext, znewcardName)
            if hresult != SCARD_S_SUCCESS:
                raise error(
                    "Failed to remove card type: " + SCardGetErrorMessage(hresult)
                )

        finally:
            hresult2 = SCardReleaseContext(hcontext)
            if hresult2 != SCARD_S_SUCCESS:
                raise error(
                    "Failed to release context: " + SCardGetErrorMessage(hresult)
                )
            print("Released context.")

    main()

elif "pcsclite" == resourceManager:
    print("SCardListInterfaces not supported by pcsc lite")

SCardListInterfaces not supported by pcsc lite


In [8]:
from smartcard.scard import *
from smartcard.util import toASCIIString, toBytes, toHexString

try:
    hresult, hcontext = SCardEstablishContext(SCARD_SCOPE_USER)
    if hresult != SCARD_S_SUCCESS:
        raise error("Failed to establish context: " + SCardGetErrorMessage(hresult))
    print("Context established!")

    try:
        hresult, readers = SCardListReaders(hcontext, [])
        if hresult != SCARD_S_SUCCESS:
            raise error("Failed to list readers: " + SCardGetErrorMessage(hresult))
        print("PCSC Readers:", readers)

        if len(readers) < 1:
            raise error("No smart card readers")

        for zreader in readers:

            print("Trying to Control reader:", zreader)

            try:
                hresult, hcard, dwActiveProtocol = SCardConnect(
                    hcontext, zreader, SCARD_SHARE_DIRECT, SCARD_PROTOCOL_T0
                )
                if hresult != SCARD_S_SUCCESS:
                    raise error("Unable to connect: " + SCardGetErrorMessage(hresult))
                print("Connected with active protocol", dwActiveProtocol)

                try:
                    if "winscard" == resourceManager:
                        # IOCTL_SMARTCARD_GET_ATTRIBUTE = SCARD_CTL_CODE(2)
                        hresult, response = SCardControl(
                            hcard,
                            SCARD_CTL_CODE(2),
                            toBytes("%.8lx" % SCARD_ATTR_VENDOR_NAME),
                        )
                        if hresult != SCARD_S_SUCCESS:
                            raise error(
                                "SCardControl failed: " + SCardGetErrorMessage(hresult)
                            )
                        print("SCARD_ATTR_VENDOR_NAME:", toASCIIString(response))
                    elif "pcsclite" == resourceManager:
                        # get feature request
                        hresult, response = SCardControl(
                            hcard, SCARD_CTL_CODE(3400), []
                        )
                        if hresult != SCARD_S_SUCCESS:
                            raise error(
                                "SCardControl failed: " + SCardGetErrorMessage(hresult)
                            )
                        print("CM_IOCTL_GET_FEATURE_REQUEST:", toHexString(response))
                finally:
                    hresult = SCardDisconnect(hcard, SCARD_UNPOWER_CARD)
                    if hresult != SCARD_S_SUCCESS:
                        raise error(
                            "Failed to disconnect: " + SCardGetErrorMessage(hresult)
                        )
                    print("Disconnected")

            except error as message:
                print(error, message)

    finally:
        hresult = SCardReleaseContext(hcontext)
        if hresult != SCARD_S_SUCCESS:
            raise error("Failed to release context: " + SCardGetErrorMessage(hresult))
        print("Released context.")

except error as e:
    print(e)

Context established!
PCSC Readers: ['Alcor Micro AU9540 00 00', 'REINER SCT cyberJack RFID basis 01 00']
Trying to Control reader: Alcor Micro AU9540 00 00
Connected with active protocol 0
CM_IOCTL_GET_FEATURE_REQUEST: 12 04 42 33 00 12
Disconnected
Trying to Control reader: REINER SCT cyberJack RFID basis 01 00
Connected with active protocol 0
CM_IOCTL_GET_FEATURE_REQUEST: 12 04 42 33 00 12
Disconnected
Released context.


In [25]:
# example1
from smartcard.Exceptions import CardConnectionException, NoCardException
from smartcard.System import *
from smartcard import util


class MustBeEvenException(Exception):
    pass


if __name__ == '__main__':

    # get and print a list of readers attached to the system
    sc_readers = readers()
    print(sc_readers)

    # create a connection to the first reader
    first_reader = sc_readers[1]
    connection = first_reader.createConnection()

    # get ready for a command
    get_uid = util.toBytes("FF CA 00 00 00")
    alt_get_uid = [0xFF, 0xCA, 0x00, 0x00, 0x00] # alternative to using the helper

    try:
        # send the command and capture the response data and status
        connection.connect()
        data, sw1, sw2 = connection.transmit(get_uid)

        # print the response
        uid = util.toHexString(data)
        status = util.toHexString([sw1, sw2])
        print("UID = {}\tstatus = {}".format(uid, status))
    except NoCardException:
        print("ERROR: Card not present")

['Alcor Micro AU9540 00 00', 'REINER SCT cyberJack RFID basis 01 00']
UID = 04 89 67 62 F7 71 80	status = 90 00


In [12]:
new_cmd = util.toBytes("3A 00 E1 00 00")

In [21]:
new_cmd = util.toBytes("1B FF FF FF FF")

In [32]:
new_cmd = util.toBytes("E0 00 00 18 00")

In [35]:
connection.transmit(new_cmd)

CardConnectionException: Card returned no valid response: Command successful. (0x00000000)

In [ ]:
connection.transmit(get_uid)

([4, 137, 103, 98, 247, 113, 128], 144, 0)